# TabICL demo

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from tabicl import TabICLClassifier

from tab_utils import plot_data_2d, plot_function_2d

## Create half-moons data

In [ ]:
# generate synthetic data
np.random.seed(55555)

x, y = make_moons(n_samples=1000, shuffle=True, noise=0.15)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

print(f"Train data shape: {x_train.shape}")
print(f"Test data shape: {x_test.shape}")

In [ ]:
# plot train data
fig, ax = plt.subplots(figsize=(5, 3.5))
plot_data_2d(
    x_train,
    y_train,
    colors=(plt.cm.Set1(1), plt.cm.Set1(0)),
    ax=ax,
)
ax.set(xlim=(-2, 3), ylim=(-1.5, 2))
ax.set_aspect("equal", adjustable="box")
ax.legend(loc="upper right")
ax.grid(color="gray", alpha=0.2, linestyle="-")
ax.set_axisbelow(True)
fig.tight_layout()

## Test TabICL model

In [ ]:
# download checkpoint and initialize model
tabicl = TabICLClassifier()
tabicl.fit(x_train, y_train)

print(tabicl.model_)

In [ ]:
# predict via in-context learning
y_prob = tabicl.predict_proba(x_test)
y_pred = y_prob.argmax(axis=1)
# y_pred = tabicl.predict(x_test)

print(f"Predictions shape: {y_pred.shape}")
print(f"Probabilities shape: {y_prob.shape}")

In [ ]:
# calculate performance metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Test accuracy: {acc:.2f}")
print(f"Test precision: {prec:.2f}")
print(f"Test recall: {rec:.2f}")
print(f"Test F1-score: {f1:.2f}")

In [ ]:
# plot test data and prediction probabilities
fig, ax = plt.subplots(figsize=(6, 3.55))
plot_data_2d(
    x_test,
    y_test,
    colors=(plt.cm.Set1(1), plt.cm.Set1(0)),
    zorder=2,
    ax=ax,
)
ax.set(xlim=(-2, 3), ylim=(-1.5, 2))  # set limits before 2D function is plotted
plot_function_2d(
    lambda x: tabicl.predict_proba(x)[:, 1],
    gridpoints=201,
    levels=(0.1, 0.5, 0.9),
    z_limits=(0, 1),
    zorder=1,
    ax=ax,
)
ax.images[0].colorbar.set_label("$p(y=1|x_1,x_2)$", rotation=-90, labelpad=18)
ax.set_aspect("equal", adjustable="box")
ax.legend(loc="upper left")
ax.grid(color="gray", alpha=0.2, linestyle="-")
ax.set_axisbelow(True)
fig.tight_layout()